In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torchvision.utils import save_image

# Use GPU if available
device = torch.device("cuda" if
torch.cuda.is_available() else "cpu")
print("using device", device)

# Download CIFAR-10 dataset
transform = transforms.Compose([
    transforms.Resize(64),
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])


dataset = datasets.CIFAR10(root='data',
download=True, transform=transform)
loader = torch.utils.data.DataLoader(dataset,
batch_size=128, shuffle=True)

# --- Generator ---
class Generator(nn.Module):
    def __init__(self):
        super().__init__()
        self.model = nn.Sequential(
            nn.ConvTranspose2d(100, 512, 4, 1, 0, bias=False),
            nn.BatchNorm2d(512),
            nn.ReLU(True),
            nn.ConvTranspose2d(512, 256, 4, 2, 1, bias=False),
            nn.BatchNorm2d(256),
            nn.ReLU(True),
            nn.ConvTranspose2d(256, 128, 4, 2, 1, bias=False),
            nn.BatchNorm2d(128),
            nn.ReLU(True),
            nn.ConvTranspose2d(128, 64, 4, 2, 1, bias=False),
               nn.BatchNorm2d(64),
            nn.ReLU(True),
            nn.ConvTranspose2d(64, 3, 4, 2, 1, bias=False),
            nn.Tanh()
        )

    def forward(self, z):
        return self.model(z)

# --- Discriminator ---
class Discriminator(nn.Module):
    def __init__(self):
        super().__init__()
        self.model = nn.Sequential(
            nn.Conv2d(3, 64, 4, 2, 1, bias=False),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(64, 128, 4, 2, 1, bias=False),
 nn.BatchNorm2d(128),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(128, 256, 4, 2, 1, bias=False),
            nn.BatchNorm2d(256),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(256, 512, 4, 2, 1, bias=False),
            nn.BatchNorm2d(512),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(512, 1, 4, 1, 0, bias=False),
            nn.Sigmoid()
        )

    def forward(self, img):
        return self.model(img).view(-1, 1).squeeze(1)

G = Generator().to(device)
D = Discriminator().to(device)

criterion = nn.BCELoss()
opt_G = optim.Adam(G.parameters(), lr=0.0002, betas=(0.5, 0.999))
opt_D = optim.Adam(D.parameters(), lr=0.0002, betas=(0.5, 0.999))

epochs = 10  # You can increase to 50 for better results

for epoch in range(epochs):
    for i, (imgs, _) in enumerate(loader):
        real_imgs = imgs.to(device)
        real_labels = torch.ones(imgs.size(0), device=device)
        fake_labels = torch.zeros(imgs.size(0), device=device)
         # Train Discriminator
        z = torch.randn(imgs.size(0), 100, 1, 1, device=device)
        fake_imgs = G(z)
        D_real = D(real_imgs)
        D_fake = D(fake_imgs.detach())
        loss_D = (criterion(D_real, real_labels) + criterion(D_fake, fake_labels)) / 2
        opt_D.zero_grad()
        loss_D.backward()
        opt_D.step()

        # Train Generator
        D_fake = D(fake_imgs)
        loss_G = criterion(D_fake, real_labels)
        opt_G.zero_grad()
        loss_G.backward()
        opt_G.step()
    print(f"Epoch [{epoch+1}/{epochs}] | D Loss: {loss_D.item():.4f} | G Loss: {loss_G.item():.4f}")

    # Save samples
    save_image(fake_imgs[:25], f"generated_epoch_{epoch+1}.png", nrow=5, normalize=True)

using device cuda
Epoch [1/10] | D Loss: 0.3499 | G Loss: 4.6000
Epoch [2/10] | D Loss: 0.3480 | G Loss: 4.5336
Epoch [3/10] | D Loss: 0.1111 | G Loss: 3.7860
Epoch [4/10] | D Loss: 0.2641 | G Loss: 3.0825
Epoch [5/10] | D Loss: 1.1649 | G Loss: 8.6036
Epoch [6/10] | D Loss: 0.0758 | G Loss: 4.2795
Epoch [7/10] | D Loss: 0.2162 | G Loss: 6.1706
Epoch [8/10] | D Loss: 0.1092 | G Loss: 3.3732
Epoch [9/10] | D Loss: 0.0547 | G Loss: 4.5574
Epoch [10/10] | D Loss: 0.0819 | G Loss: 5.0827


In [ ]:
!pip install torch torchvision matplotlib